In [2]:

# If "Suisse Int’l" is the exact internal name, this should work:
# (Optional) You can explicitly register each font file with Matplotlib:
import matplotlib.font_manager as fm
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-Light.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-LightItalic.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-Medium.ttf")
fm.fontManager.addfont("/Users/vaienti/Library/Fonts/SuisseIntl-MediumItalic.ttf")


import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

import numpy as np


colors = {
    'pink_dark' : '#f75785',
    'pink_light' : '#f8b0be',
    'aqua_dark' : '#009da5',
    'aqua_light' : '#3cc5be',
    'orange_dark' : '#ffa631',
    'orange_light' : '#ffd766',
    'red_dark' : '#e84743',
    'red_light' : '#ed8e83',
    'blue_dark': '#489fee',
    'blue_light': '#8fcfff',
    'dark_grey': '#413d3a',
    'light_grey': '#cac7c7',
}
def create_custom_cmap(colors):
    """
    Creates a custom colormap that transitions:
      #FB4C59 -> #f2a02a -> #009da5
    and returns it as a LinearSegmentedColormap.
    """
    # Define our three anchor colors in hex
    sel_colors = [colors['aqua_dark'], colors['orange_dark'], colors['pink_dark']]
    # Create a colormap with these three points
    cmap = mcolors.LinearSegmentedColormap.from_list("my_custom_cmap", sel_colors)
    return cmap

colors_list = list(colors.values())



color_map = create_custom_cmap(colors)
x = np.linspace(0, 1, 100)
X, Y = np.meshgrid(x, x)
Z = X + Y


## 1. Import Target and Anchor maps

Later: we will put two or three example maps in a folder, for now we do the import from the pc. For the moment we have two lists of maps and link to their elements inside the folder input.

In [3]:
# read the list of anchor and target maps

import pickle
with open('./input/anchor_maps.pkl', 'rb') as f:
    anchor_maps = pickle.load(f)
with open('./input/target_maps.pkl', 'rb') as f:
    target_maps = pickle.load(f)



In [4]:
# create from scratch the MapDataset object
from modules.MapDataset import MapDataset, create_map_dataset
from tqdm import tqdm

# Creating a list of MapDataset objects from the collected map data with tqdm
anchor_map_objects_list = [create_map_dataset(map_data) for map_data in tqdm(anchor_maps[:10], desc="Creating MapDataset objects")]

target_map_objects_list = [create_map_dataset(map_data) for map_data in tqdm(target_maps[:10], desc="Creating MapDataset objects")]  



Loaded SuperPoint model


Creating MapDataset objects:   0%|          | 0/10 [00:00<?, ?it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1898_schneller' with author 'schneller' and year '1898'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1898_schneller/schneller_1898.jpeg
libpng warning: iCCP: known incorrect sRGB profile
INFO:modules.MapDataset:Loaded mask from /Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/1898_schneller/schneller_1898_mask.png
INFO:modules.MapDataset:MapDataset created successfully for folder: 1898_schneller
Creating MapDataset objects:  10%|█         | 1/10 [00:00<00:02,  3.56it/s]INFO:modules.MapDataset:Creating MapDataset for folder '1860_weller' with author 'weller' and year '1860'.
INFO:modules.MapDataset:Loaded image from /Users/vaienti/Li

In [5]:
base_dataset = anchor_map_objects_list
target_dataset = target_map_objects_list

## 2. First Pairwise Matching

### 2.1 BaseMaps: Orient North and Generate Tensors

In [6]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

In [10]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import importlib
import modules.MapDataset
import modules.homologous_points_detection
importlib.reload(modules.MapDataset)


# Process the base_dataset:   
for map_obj in tqdm(base_dataset, desc=f"Base Processing"):
    map_obj.calculate_and_store_north_rotation()
    map_obj.run_superpoint_pipeline()

Loaded SuperGlue model ("outdoor" weights)


Base Processing: 100%|██████████| 10/10 [00:38<00:00,  3.81s/it]


### 2.2 AnchorMaps: Find Best Match with BaseMaps, Orient North and Generate Tensors

In [17]:
importlib.reload(modules.homologous_points_detection)
importlib.reload(modules.MapDataset)
from modules.MapDataset import MapDataset
from modules.homologous_points_detection import find_best_matches

Loaded SuperGlue model ("outdoor" weights)


In [15]:
from modules.homologous_points_detection import find_single_best_match, estimate_north_rotation
import cv2
import numpy as np
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
from skimage.measure import ransac
from skimage.transform import AffineTransform
import copy
import pickle

    
# Lop over each target map object in the current fold.
for map_obj in tqdm(target_dataset, desc=f"Target Maps Processing"):
    # Run the SuperPoint pipeline on the target map.
    map_obj.run_superpoint_pipeline()
    best_match = find_single_best_match(map_obj, base_dataset)
    num_matches = len(best_match['superglue_matches_df'])
    initial_num_matches = num_matches
        
    min_match_score = 0.3
    rotation_angles = [np.pi / 2, np.pi, 3 * np.pi / 2]
    rotation_degrees = [90, 180, 270]
        
        # If the current number of matches is less than 100, attempt imposed rotations.
    if num_matches < 100:
        best_num_matches = num_matches
        best_angle_rad = None

        for angle_rad, angle_deg in zip(rotation_angles, rotation_degrees):
            map_obj.north_rotation_angle = angle_rad
            map_obj.run_superpoint_pipeline()
            best_match_rotated = find_single_best_match(map_obj, base_dataset)
            new_num_matches = len(best_match_rotated['superglue_matches_df'])
            if new_num_matches > best_num_matches:
                best_num_matches = new_num_matches
                best_angle_rad = angle_rad
                best_match = best_match_rotated

        if best_angle_rad is not None:
            map_obj.north_rotation_angle = best_angle_rad
            map_obj.run_superpoint_pipeline()
            best_match = best_match_rotated
            #print(f"Imposed rotation of {np.degrees(best_angle_rad):.2f}° improved matches from {num_matches} to {best_num_matches}.")
            derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=True)
            #print('Derived north rotation (deg):', np.degrees(derived_north_rotation))
        else:
            map_obj.north_rotation_angle = 0.0
            map_obj.run_superpoint_pipeline()
            best_match = find_single_best_match(map_obj, base_dataset)
            derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=False)
            #print(f"No imposed rotation improved matches from {num_matches}. Calculated north rotation: {np.degrees(derived_north_rotation):.2f}°.")
    else:
        # If there are enough matches, directly estimate the north rotation.
        derived_north_rotation = estimate_north_rotation(map_obj, best_match, min_match_score=min_match_score, plot_transformation=False)

    map_obj.north_rotation_angle = derived_north_rotation
    map_obj.run_superpoint_pipeline()





Target Maps Processing: 100%|██████████| 10/10 [02:17<00:00, 13.71s/it]


### 2.3 Repeat Pairwise Matching with the new tensors
We compare each map of the target dataset with each map of the anchor dataset and store all the matches. 

In [18]:
from modules.homologous_points_detection import  find_best_matches
from modules.MapDataset import MapDataset
from typing import List

#in Evaluation mode we can collect more than one match per map:
find_best_matches(target_dataset, base_dataset, threshold_number_matches=100, number_best_results=3, evaluation = True, min_score=0.1 )


Evaluating maps: 100%|██████████| 10/10 [00:42<00:00,  4.25s/it]


## 3. Match Improvements

### 3.1 Ransac + Delaunay Cleaning

### 3.2 Points Addition

### 3.3 Ransac + Delaunay Cleaning

## 4. GCP transfer


## 5. Results Reliability

## Saving the results: go over each folder of the target maps, create a folder and save the results.